<a href="https://colab.research.google.com/github/timurios/datasets/blob/master/mistral7_photonable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip uninstall -y bitsandbytes
!pip install -U pip

!pip install -U \
    bitsandbytes>=0.46.1 \
    transformers \
    accelerate \
    peft \
    trl \
    datasets \
    sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [9]:
import torch
import bitsandbytes as bnb

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("bitsandbytes:", bnb.__version__)

CUDA: True
GPU: Tesla T4
bitsandbytes: 0.50.2


In [10]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# T4 -> explicitly use FP16
COMPUTE_DTYPE = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Loading Mistral...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.config.use_cache = False

print("✓ Mistral loaded")
print("Device:", model.device)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✓ Mistral loaded
Device: cuda:0


In [11]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

OUTPUT_DIR = "/content/photonable-mistral-7b-lora"

MAX_SEQ_LENGTH = 2048

In [12]:
from huggingface_hub import notebook_login

notebook_login()

In [14]:
free, total = torch.cuda.mem_get_info()

print(f"Total VRAM: {total / 1024**3:.2f} GB")
print(f"Free VRAM:  {free / 1024**3:.2f} GB")
print(f"Used VRAM:  {(total-free) / 1024**3:.2f} GB")

Total VRAM: 14.56 GB
Free VRAM:  10.46 GB
Used VRAM:  4.10 GB


In [15]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

print("✓ Model prepared for QLoRA")

✓ Model prepared for QLoRA


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


In [ ]:
import torch
import random
import math
import json
import os
import numpy as np

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Model type:", type(model))
print("Tokenizer:", type(tokenizer))

model.print_trainable_parameters()

CUDA: True
GPU: Tesla T4
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Tokenizer: <class 'transformers.tokenization_utils_tokenizers.TokenizersBackend'>
trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_DIR = "/content/drive/MyDrive/photonics_mistral_v01"
os.makedirs(RUN_DIR, exist_ok=True)

SEED = 20261001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

COUNTS = {
    "conceptual": 300,
    "numerical": 240,
    "pic_design": 180,
    "maxwell": 120,
    "solver_selection": 120,
    "debugging": 120,
    "quantum_nonlinear": 60,
    "photonable_tools": 60,
}

print("Total examples:", sum(COUNTS.values()))

Mounted at /content/drive
Total examples: 1200


In [ ]:
C = 299_792_458.0
H = 6.62607015e-34
E_CHARGE = 1.602176634e-19

records = []

def add_record(question, answer, category, source_type):
    records.append({
        "question": question.strip(),
        "answer": answer.strip(),
        "category": category,
        "source_type": source_type,
    })

def rnd(a, b, digits=3):
    return round(random.uniform(a, b), digits)

In [ ]:
concept_bank = [
    (
        "effective index",
        "The effective index n_eff relates a guided mode's propagation constant to the free-space wavenumber through beta = n_eff k0. It lies between values determined by the core and surrounding media and summarizes how the modal field experiences the structure."
    ),
    (
        "group index",
        "The group index n_g describes the propagation of the envelope of a wave packet. It is related to group velocity by v_g = c/n_g and includes material and waveguide dispersion."
    ),
    (
        "guided mode",
        "A guided mode is a self-consistent electromagnetic field distribution that propagates along a structure while remaining transversely confined. Its propagation constant and field profile satisfy Maxwell's equations together with the boundary conditions."
    ),
    (
        "evanescent field",
        "An evanescent field decays approximately exponentially away from an interface or guided structure rather than carrying substantial power normal to the interface. Evanescent overlap enables waveguide coupling, sensing, and frustrated total internal reflection."
    ),
    (
        "total internal reflection",
        "Total internal reflection occurs when light travels from a higher-index medium toward a lower-index medium at an incidence angle greater than the critical angle. No propagating transmitted plane wave exists in the lower-index medium, although an evanescent field remains."
    ),
    (
        "waveguide dispersion",
        "Waveguide dispersion occurs because the modal field distribution and effective index change with wavelength. It can significantly modify total dispersion even when the constituent material dispersion is fixed."
    ),
    (
        "group velocity dispersion",
        "Group velocity dispersion describes the frequency dependence of group velocity. It is commonly represented by beta_2 = d^2 beta/d omega^2 and controls temporal pulse broadening and nonlinear phase-matching behavior."
    ),
    (
        "TE and TM modes",
        "TE-like and TM-like labels describe the dominant polarization of guided modes. In high-index-contrast integrated waveguides the fields are fully vectorial, so the labels usually indicate dominant rather than perfectly transverse components."
    ),
    (
        "mode cutoff",
        "Mode cutoff is the point at which a guided mode ceases to remain bound as geometry, frequency, or refractive index changes. Below or above the relevant cutoff condition the field can become radiative instead of guided."
    ),
    (
        "single-mode waveguide",
        "A single-mode waveguide supports only one guided spatial mode for the specified polarization and wavelength range. Single-mode operation simplifies interference, coupling, and circuit behavior."
    ),
    (
        "mode confinement",
        "Mode confinement describes how strongly electromagnetic energy is localized in a desired region. Strong confinement generally enables smaller bends and stronger interactions but can increase sensitivity to fabrication roughness."
    ),
    (
        "ring resonance",
        "A ring resonance occurs when the accumulated round-trip phase is an integer multiple of 2 pi. Approximately, beta L = 2 pi m, where L is the ring circumference and m is an integer."
    ),
    (
        "free spectral range",
        "The free spectral range is the frequency or wavelength spacing between neighboring resonances. For a ring resonator, the frequency FSR is approximately c/(n_g L)."
    ),
    (
        "critical coupling",
        "Critical coupling occurs when coupling loss from a resonator into the bus matches the resonator's intrinsic round-trip loss. For an ideal single-bus resonator this can produce very deep transmission extinction at resonance."
    ),
    (
        "intrinsic and loaded Q",
        "Intrinsic Q characterizes losses internal to a resonator, while loaded Q includes both intrinsic losses and external coupling. Their inverse quality factors approximately add: 1/Q_loaded = 1/Q_intrinsic + 1/Q_coupling."
    ),
    (
        "Mach-Zehnder interferometer",
        "A Mach-Zehnder interferometer splits a field into two paths and recombines them. A differential phase shift changes constructive and destructive interference at its outputs, allowing modulation, switching, filtering, and sensing."
    ),
    (
        "directional coupler",
        "A directional coupler transfers optical power between nearby waveguides through evanescent coupling. The coupled structure can be described using even and odd supermodes whose propagation-constant difference determines the coupling length."
    ),
    (
        "supermode",
        "A supermode is an eigenmode of the complete coupled structure rather than of either isolated waveguide. Symmetric coupled waveguides commonly produce even and odd supermodes with different propagation constants."
    ),
    (
        "MMI coupler",
        "A multimode-interference coupler uses interference between several modes in a wider multimode section to reproduce input-field images at selected propagation distances. It can provide relatively fabrication-tolerant splitting and combining."
    ),
    (
        "Bragg grating",
        "A Bragg grating uses a periodic refractive-index or geometry perturbation to coherently couple waves. First-order reflection is approximately centered near lambda_B = 2 n_eff Lambda."
    ),
    (
        "grating coupler",
        "A grating coupler uses periodic perturbations to provide momentum matching between a guided mode and a radiated mode, commonly coupling light between a photonic chip and an optical fiber."
    ),
    (
        "photonic bandgap",
        "A photonic bandgap is a frequency interval in which propagating electromagnetic Bloch modes are absent for specified directions or polarizations. It arises from periodic dielectric structure rather than electronic band physics."
    ),
    (
        "Bloch mode",
        "A Bloch mode is an eigenmode of a periodic electromagnetic structure and has the form of a periodic function multiplied by a phase factor exp(i k dot r)."
    ),
    (
        "propagation loss",
        "Propagation loss describes reduction of guided optical power with distance. Contributions can include material absorption, sidewall scattering, radiation, substrate leakage, and imperfections."
    ),
    (
        "scattering loss",
        "Scattering loss occurs when imperfections such as surface or sidewall roughness couple energy from a desired guided mode into radiation or unwanted modes."
    ),
    (
        "PML",
        "A perfectly matched layer is an artificial absorbing boundary designed to suppress reflections from the edge of a computational domain. A well-configured PML approximates an open electromagnetic domain."
    ),
    (
        "S-parameters",
        "Scattering parameters describe how modal wave amplitudes entering a multiport device are transformed into outgoing modal amplitudes. For example, S21 describes transmission from port 1 to port 2 under the specified normalization."
    ),
    (
        "reciprocity",
        "Electromagnetic reciprocity relates source and response configurations in linear reciprocal media. Magneto-optic bias, time modulation, or other nonreciprocal mechanisms can break the usual reciprocal relationship."
    ),
    (
        "phase matching",
        "Phase matching requires interacting waves to maintain an appropriate relationship between their propagation constants so that generated fields accumulate coherently rather than repeatedly cancelling."
    ),
    (
        "Kerr effect",
        "The optical Kerr effect is an intensity-dependent refractive-index response often approximated as n = n0 + n2 I. It enables self-phase modulation, four-wave mixing, nonlinear switching, and frequency-comb physics."
    ),
    (
        "second-harmonic generation",
        "Second-harmonic generation is a chi(2) nonlinear process in which two photons at angular frequency omega contribute to a field at 2 omega. Efficient conversion requires suitable nonlinear tensor components and phase matching."
    ),
    (
        "mode volume",
        "Optical mode volume measures the spatial localization of an electromagnetic resonance. Small mode volume increases the local field per photon and is important for light-matter interaction."
    ),
    (
        "Purcell enhancement",
        "Purcell enhancement is the modification of spontaneous-emission rate by the electromagnetic environment. Near a resonant cavity, high Q and small mode volume can strongly enhance emission into the cavity mode under suitable spectral, spatial, and polarization matching."
    ),
    (
        "birefringence",
        "Birefringence means different polarizations experience different refractive or effective indices. It may arise from material anisotropy, stress, or asymmetric waveguide geometry."
    ),
    (
        "adiabatic taper",
        "An adiabatic taper changes geometry gradually enough that power remains predominantly in the desired evolving eigenmode rather than coupling strongly to radiation or other modes."
    ),
]

concept_prompt_styles = [
    "Explain {concept} in integrated photonics.",
    "What is {concept}, and why does it matter in photonic devices?",
    "Give a concise graduate-level explanation of {concept}.",
    "Explain the physical meaning of {concept} rather than only defining it.",
    "How should a photonics engineer think about {concept}?",
    "Describe {concept} and one reason it matters in simulation.",
    "Explain {concept} in terms of electromagnetic fields.",
    "What does {concept} mean in an optical waveguide or PIC?",
]

for i in range(COUNTS["conceptual"]):
    concept, answer = concept_bank[i % len(concept_bank)]
    style = concept_prompt_styles[(i // len(concept_bank)) % len(concept_prompt_styles)]

    add_record(
        style.format(concept=concept),
        answer,
        "conceptual",
        "authored_standard_photonics"
    )

random.shuffle(records)

print("Conceptual added:", COUNTS["conceptual"])

Conceptual added: 300


In [ ]:
def numerical_example():
    kind = random.choice([
        "ring_fsr",
        "rayleigh",
        "photon_energy",
        "fresnel",
        "fiber_na",
        "fiber_v",
        "q_linewidth",
        "mzi_phase",
        "loss",
        "critical_angle",
        "brewster",
        "coupler"
    ])

    if kind == "ring_fsr":
        R_um = rnd(5, 100, 2)
        ng = rnd(1.7, 4.5, 3)

        R_m = R_um * 1e-6
        fsr_hz = C / (ng * 2 * math.pi * R_m)
        fsr_ghz = fsr_hz / 1e9

        q = (
            f"A microring has radius {R_um} um and group index {ng}. "
            "Estimate its frequency free spectral range in GHz."
        )

        a = (
            "Use FSR = c/(n_g L), with L = 2*pi*R.\n"
            f"L = {2*math.pi*R_um:.3f} um.\n"
            f"FSR = {fsr_ghz:.3f} GHz."
        )

    elif kind == "rayleigh":
        w_um = rnd(1.0, 20.0, 2)
        lam_nm = random.choice([532, 633, 780, 1064, 1310, 1550])

        w = w_um * 1e-6
        lam = lam_nm * 1e-9

        zR = math.pi * w**2 / lam

        q = (
            f"A Gaussian beam has waist radius {w_um} um at {lam_nm} nm. "
            "Calculate its Rayleigh range."
        )

        a = (
            "For a Gaussian beam, z_R = pi*w_0^2/lambda.\n"
            f"z_R = {zR*1e3:.4f} mm."
        )

    elif kind == "photon_energy":
        lam_nm = random.randint(400, 2000)
        E_eV = H * C / (lam_nm * 1e-9) / E_CHARGE

        q = f"What is the photon energy at wavelength {lam_nm} nm in eV?"

        a = (
            "Use E = hc/lambda.\n"
            f"E = {E_eV:.4f} eV."
        )

    elif kind == "fresnel":
        n1 = rnd(1.0, 2.0, 3)
        n2 = rnd(2.1, 4.0, 3)

        R = ((n1 - n2) / (n1 + n2))**2

        q = (
            f"At normal incidence, light passes from n1={n1} to n2={n2}. "
            "Ignoring absorption, calculate the power reflectance."
        )

        a = (
            "At normal incidence, R = |(n1-n2)/(n1+n2)|^2.\n"
            f"R = {R:.5f}, or {100*R:.3f}%."
        )

    elif kind == "fiber_na":
        ncore = rnd(1.44, 1.50, 4)
        delta = rnd(0.002, 0.025, 4)
        nclad = ncore - delta

        NA = math.sqrt(max(ncore**2 - nclad**2, 0))

        q = (
            f"A step-index fiber has n_core={ncore} and n_clad={nclad:.4f}. "
            "Calculate its numerical aperture in air."
        )

        a = (
            "For a step-index fiber in air, "
            "NA = sqrt(n_core^2 - n_clad^2).\n"
            f"NA = {NA:.4f}."
        )

    elif kind == "fiber_v":
        radius_um = rnd(1.5, 25, 2)
        NA = rnd(0.05, 0.35, 3)
        lam_um = rnd(0.6, 1.7, 3)

        V = 2 * math.pi * radius_um * NA / lam_um

        q = (
            f"A fiber has core radius {radius_um} um, NA={NA}, "
            f"and wavelength {lam_um} um. Calculate its normalized V number."
        )

        a = (
            "Use V = 2*pi*a*NA/lambda.\n"
            f"V = {V:.4f}."
        )

    elif kind == "q_linewidth":
        lam_nm = random.choice([780, 1064, 1310, 1550])
        Q = random.randint(5_000, 2_000_000)

        delta_nm = lam_nm / Q

        q = (
            f"A cavity resonates at {lam_nm} nm with Q={Q}. "
            "Estimate its wavelength linewidth."
        )

        a = (
            "For a narrow resonance, Q approximately equals lambda/delta_lambda.\n"
            f"delta_lambda = {delta_nm:.6f} nm "
            f"= {delta_nm*1000:.3f} pm."
        )

    elif kind == "mzi_phase":
        neff = rnd(1.5, 3.5, 3)
        dL_um = rnd(1, 100, 2)
        lam_nm = random.choice([780, 1064, 1310, 1550])

        phase = 2 * math.pi * neff * dL_um / (lam_nm / 1000)

        q = (
            f"An MZI has effective index {neff}, path-length difference "
            f"{dL_um} um, and wavelength {lam_nm} nm. "
            "Ignoring dispersion, calculate the phase difference in radians."
        )

        a = (
            "Use delta_phi = 2*pi*n_eff*delta_L/lambda.\n"
            f"delta_phi = {phase:.4f} rad."
        )

    elif kind == "loss":
        alpha = rnd(0.1, 10, 3)
        L_cm = rnd(0.1, 10, 2)

        total_db = alpha * L_cm
        T = 10**(-total_db / 10)

        q = (
            f"A waveguide has propagation loss {alpha} dB/cm and length "
            f"{L_cm} cm. What fraction of optical power remains?"
        )

        a = (
            f"Total loss = {total_db:.4f} dB.\n"
            "Power transmission is T = 10^(-loss_dB/10).\n"
            f"T = {T:.5f}, or {100*T:.3f}%."
        )

    elif kind == "critical_angle":
        n1 = rnd(1.5, 3.5, 3)
        n2 = rnd(1.0, n1 - 0.05, 3)

        angle = math.degrees(math.asin(n2/n1))

        q = (
            f"Light travels from n1={n1} into n2={n2}. "
            "Calculate the critical angle for total internal reflection."
        )

        a = (
            "theta_c = arcsin(n2/n1).\n"
            f"theta_c = {angle:.3f} degrees."
        )

    elif kind == "brewster":
        n1 = rnd(1.0, 1.6, 3)
        n2 = rnd(1.7, 3.5, 3)

        angle = math.degrees(math.atan(n2/n1))

        q = (
            f"An interface has n1={n1} and n2={n2}. "
            "For nonmagnetic isotropic media, estimate the Brewster angle "
            "for p polarization."
        )

        a = (
            "theta_B = arctan(n2/n1).\n"
            f"theta_B = {angle:.3f} degrees."
        )

    else:
        beta_even = rnd(8, 15, 4)
        dbeta = rnd(0.02, 0.4, 4)
        beta_odd = beta_even - dbeta

        L3 = math.pi / (2 * abs(beta_even - beta_odd))

        q = (
            f"A symmetric directional coupler has beta_even={beta_even} rad/um "
            f"and beta_odd={beta_odd:.4f} rad/um. "
            "Estimate the interaction length for ideal 50:50 coupling."
        )

        a = (
            "For ideal excitation of two equal supermodes, the cross-port "
            "power varies as sin^2(delta_beta*L/2). "
            "For 50:50 coupling, delta_beta*L/2 = pi/4.\n"
            "Thus L_3dB = pi/(2*|beta_even-beta_odd|).\n"
            f"L_3dB = {L3:.4f} um."
        )

    return q, a


for _ in range(COUNTS["numerical"]):
    q, a = numerical_example()

    add_record(
        q,
        a,
        "numerical",
        "synthetic_formula_ground_truth"
    )

print("Numerical added:", COUNTS["numerical"])

Numerical added: 240


In [ ]:
def pic_design_example():

    kind = random.choice([
        "ring",
        "mzi",
        "directional_coupler",
        "bragg",
        "phase_shifter",
        "taper",
        "mmi",
        "grating_coupler"
    ])

    lam_nm = random.choice([1310, 1550])

    if kind == "ring":
        ng = rnd(1.8, 4.2, 2)
        fsr_nm = rnd(2, 20, 2)

        lam_um = lam_nm / 1000
        fsr_um = fsr_nm / 1000

        R_um = lam_um**2 / (2 * math.pi * ng * fsr_um)

        q = (
            f"Give an initial design strategy for a microring near {lam_nm} nm "
            f"with group index about {ng} and target FSR {fsr_nm} nm."
        )

        a = (
            "Start from the resonance-spacing approximation "
            "delta_lambda approximately lambda^2/(n_g*2*pi*R).\n"
            f"This gives an initial radius R approximately {R_um:.2f} um.\n"
            "Then solve the actual waveguide eigenmode to obtain wavelength-dependent "
            "n_eff and n_g, update the radius, choose the bus-ring gap from a coupling "
            "simulation, and verify resonance position, extinction, insertion loss, "
            "and loaded Q with a wavelength sweep."
        )

    elif kind == "mzi":
        ng = rnd(1.8, 4.2, 2)
        fsr_nm = rnd(5, 40, 2)

        lam_um = lam_nm/1000
        fsr_um = fsr_nm/1000

        dL = lam_um**2 / (ng * fsr_um)

        q = (
            f"Design an initial unbalanced MZI at {lam_nm} nm with "
            f"target spectral period around {fsr_nm} nm and n_g about {ng}."
        )

        a = (
            "For a first estimate use delta_lambda approximately "
            "lambda^2/(n_g*delta_L).\n"
            f"An initial path difference is therefore about {dL:.2f} um.\n"
            "Use validated splitters/combiners, calculate eigenmodes of the arms, "
            "include wavelength-dependent phase and loss, then sweep wavelength "
            "to verify the actual transfer function."
        )

    elif kind == "directional_coupler":
        q = (
            f"How would you design a 3 dB directional coupler around {lam_nm} nm?"
        )

        a = (
            "Choose waveguide cross sections that support the desired input mode, "
            "then parameterize the coupling gap. Solve the coupled cross section "
            "for even and odd supermodes. Their propagation-constant difference "
            "provides an initial 3 dB coupling length "
            "L approximately pi/(2*|beta_even-beta_odd|). "
            "Then simulate the complete structure including approach bends and "
            "tapers, extract modal S-parameters, and optimize gap and interaction "
            "length for |S21|^2 approximately |S31|^2 approximately 0.5."
        )

    elif kind == "bragg":
        neff = rnd(1.5, 3.5, 2)
        period_nm = lam_nm/(2*neff)

        q = (
            f"Give a first-order Bragg-grating starting point for {lam_nm} nm "
            f"if n_eff is approximately {neff}."
        )

        a = (
            "Use the first-order Bragg condition lambda_B approximately "
            "2*n_eff*Lambda.\n"
            f"Initial period Lambda approximately {period_nm:.2f} nm.\n"
            "This is only a starting estimate because the periodic structure changes "
            "the modal effective index. Use an eigenmode/periodic solver or full-wave "
            "simulation to refine the period, modulation depth, number of periods, "
            "bandwidth, and reflection."
        )

    elif kind == "phase_shifter":
        dn = rnd(0.0002, 0.01, 5)
        L_um = (lam_nm/1000) / (2*dn)

        q = (
            f"An index-control mechanism provides delta_n approximately {dn}. "
            f"Estimate the interaction length required for a pi phase shift at {lam_nm} nm."
        )

        a = (
            "The differential phase is delta_phi = 2*pi*delta_n*L/lambda. "
            "Setting delta_phi=pi gives L_pi=lambda/(2*delta_n).\n"
            f"L_pi approximately {L_um:.2f} um.\n"
            "A real design should also include mode-overlap, loss, dispersion, "
            "electrode or heater geometry, and thermal/electrical constraints."
        )

    elif kind == "taper":
        q = (
            f"Design a low-reflection waveguide width taper operating near {lam_nm} nm."
        )

        a = (
            "First solve local eigenmodes over the width range. Make the geometry "
            "change slowly enough that the desired mode evolves adiabatically and "
            "has weak coupling to competing guided or radiation modes. "
            "Parameterize taper length and profile, then use EME or full-wave "
            "simulation to monitor insertion loss, reflection, radiation, and "
            "higher-order-mode excitation. Increase length until the required "
            "performance converges."
        )

    elif kind == "mmi":
        q = (
            f"Give a simulation workflow for designing a 1x2 MMI splitter near {lam_nm} nm."
        )

        a = (
            "Select single-mode access waveguides and parameterize the multimode-section "
            "width and length. Compute the modes of the multimode section, estimate "
            "self-imaging lengths, then sweep width, length, and output positions. "
            "Evaluate modal transmission into both output waveguides, imbalance, "
            "excess loss, phase error, bandwidth, and fabrication sensitivity. "
            "Use full-wave or EME validation for the final geometry."
        )

    else:
        q = (
            f"Give a practical workflow for designing a fiber grating coupler near {lam_nm} nm."
        )

        a = (
            "Determine the chip waveguide mode and target fiber mode first. "
            "Use the grating equation to obtain an initial period for the chosen "
            "radiation angle and effective index. Parameterize period, duty cycle, "
            "etch depth, number of periods, and apodization. Use a 2D sweep for "
            "initial optimization when appropriate, then verify the final structure "
            "with a 3D full-wave model and evaluate coupling efficiency, back-reflection, "
            "bandwidth, polarization sensitivity, and alignment tolerance."
        )

    return q, a


for _ in range(COUNTS["pic_design"]):
    q, a = pic_design_example()

    add_record(
        q,
        a,
        "pic_design",
        "authored_photonic_design_rules"
    )

print("PIC design added:", COUNTS["pic_design"])

PIC design added: 180


In [ ]:
maxwell_bank = [
    (
        "Why can the divergence of B not produce a magnetic-monopole source term in ordinary Maxwell electromagnetism?",
        "Maxwell's equation div B = 0 expresses the absence of observed magnetic monopole charge in conventional electromagnetism. Magnetic field lines therefore have no beginning or end and form closed or continuing flux lines."
    ),
    (
        "What boundary condition applies to tangential E across an ordinary dielectric interface with no singular magnetic current?",
        "The tangential component of E is continuous across the interface: n x (E2-E1)=0, assuming no singular magnetic-current sheet."
    ),
    (
        "What happens to normal D across an interface carrying free surface charge density rho_s?",
        "The normal displacement field has a discontinuity equal to the free surface charge: n dot (D2-D1)=rho_s."
    ),
    (
        "What happens to tangential H across an interface with free surface-current density K?",
        "The tangential magnetic field satisfies n x (H2-H1)=K for a free surface-current sheet."
    ),
    (
        "Why must a high-index-contrast waveguide generally be treated with vector electromagnetic modes?",
        "Strong index contrast produces significant longitudinal and cross-polarized field components, so a purely scalar approximation can miss polarization coupling, boundary behavior, and accurate effective indices."
    ),
    (
        "What does the real part of the Poynting vector represent for harmonic fields?",
        "For harmonic fields, one-half the real part of E cross H* gives the time-averaged electromagnetic power-flow density under the usual phasor convention."
    ),
    (
        "Why can an evanescent field exist even though it does not represent a propagating wave normal to the interface?",
        "The normal wavevector component becomes imaginary, producing exponential spatial decay. The field still satisfies Maxwell's equations and can transport energy parallel to the interface or couple to another nearby structure."
    ),
    (
        "Why does mesh convergence matter in a Maxwell simulation?",
        "A numerical Maxwell solver approximates continuous fields on a finite representation. If the result changes substantially as the discretization is refined, discretization error is still significant and the reported solution is not demonstrably converged."
    ),
    (
        "What physical effect does a positive imaginary part of refractive index represent under the common exp(-i omega t) convention?",
        "Under the common exp(-i omega t) convention, a positive extinction coefficient corresponds to attenuation and therefore material absorption."
    ),
    (
        "Why is anisotropic permittivity represented as a tensor?",
        "In an anisotropic medium the electric displacement need not be parallel to the electric field. A tensor relation D = epsilon dot E is therefore required to describe direction-dependent response and polarization coupling."
    ),
    (
        "What is the purpose of periodic electromagnetic boundary conditions?",
        "Periodic or Bloch boundary conditions represent fields in a periodically repeated structure. Bloch conditions include the phase accumulated between neighboring unit cells for a specified Bloch wavevector."
    ),
    (
        "Why does a resonant cavity field build up strongly near resonance?",
        "Near resonance, successive round trips return with nearly the same phase and interfere constructively. The steady-state amplitude is limited by coupling and loss."
    ),
    (
        "Why can reducing wavelength require a finer FDTD mesh for the same material structure?",
        "The numerical grid must resolve the shortest relevant wavelength inside the material, approximately lambda0/n, as well as small geometric features. Shorter wavelengths therefore generally demand smaller grid spacing."
    ),
    (
        "Why can subpixel smoothing improve a dielectric-grid simulation?",
        "A staircase representation of curved or oblique interfaces creates geometric discretization error. Subpixel or effective-interface treatments approximate the material boundary more accurately without requiring an equally large increase in grid resolution."
    ),
    (
        "Why is power conservation a useful diagnostic in a passive photonic simulation?",
        "For a passive structure, appropriately normalized incident power should equal reflected, transmitted, absorbed, and radiated power within numerical error. A large unexplained imbalance often indicates normalization, monitor, boundary, or convergence problems."
    ),
    (
        "What distinguishes a frequency-domain Maxwell solution from a time-domain solution?",
        "A frequency-domain method solves directly for fields at one or more specified frequencies, while a time-domain method evolves fields in time and can obtain broadband spectral information by Fourier transforming the response."
    ),
]

maxwell_styles = [
    "{q}",
    "Reason through this electromagnetic question: {q}",
    "Give a physically grounded answer: {q}",
    "Answer using Maxwell-equation reasoning: {q}",
]

for i in range(COUNTS["maxwell"]):
    q0, a = maxwell_bank[i % len(maxwell_bank)]
    style = maxwell_styles[(i // len(maxwell_bank)) % len(maxwell_styles)]

    add_record(
        style.format(q=q0),
        a,
        "maxwell",
        "authored_maxwell_reasoning"
    )

print("Maxwell added:", COUNTS["maxwell"])

Maxwell added: 120


In [ ]:
solver_cases = [
    (
        "I need n_eff and the field profile of the TE0 mode of a uniform silicon waveguide.",
        "eigenmode solver",
        "The structure is invariant along the propagation direction and the requested quantities are modal eigenvalues and eigenfields."
    ),
    (
        "I need the photonic band structure of an infinite periodic photonic crystal.",
        "plane-wave expansion / periodic eigenmode solver",
        "The task is an eigenvalue problem in a periodic unit cell, making a Bloch-mode or plane-wave-expansion method appropriate."
    ),
    (
        "I need broadband reflection and transmission of a compact wavelength-scale discontinuity.",
        "FDTD",
        "A short-pulse time-domain simulation can efficiently provide broadband spectral response and full fields for a compact structure."
    ),
    (
        "I need a high-accuracy monochromatic field solution in a complicated curved resonator geometry.",
        "FEM",
        "Finite-element methods handle complex curved geometries and adaptive meshes well for frequency-domain solutions."
    ),
    (
        "I need propagation through a millimeter-long waveguide whose cross section changes gradually.",
        "BPM or EME",
        "A full 3D FDTD calculation would usually be inefficient. Propagation or eigenmode-expansion methods exploit the predominantly forward and slowly varying nature of the problem."
    ),
    (
        "I need the coupling length of two parallel uniform waveguides.",
        "coupled-waveguide eigenmode / supermode analysis",
        "The even and odd supermode propagation constants directly provide the beating and coupling length."
    ),
    (
        "I have a circuit containing hundreds of characterized MZIs, rings, and couplers and only need port-level spectra.",
        "circuit / scattering-matrix solver",
        "Using compact component S-parameters avoids an impractical full-wave simulation of the complete circuit."
    ),
    (
        "I need diffraction efficiencies of an infinite layered periodic grating.",
        "RCWA / Fourier modal method",
        "RCWA is particularly well suited to structures periodic in-plane and stratified along another direction."
    ),
    (
        "I need to model optical heating and the resulting thermo-optic index shift.",
        "coupled electromagnetic and heat solver",
        "The optical solver provides absorbed-power density, while a thermal solver determines temperature and the material properties are updated through thermo-optic coupling."
    ),
    (
        "I need to identify which guided modes are present at an output monitor.",
        "eigenmode decomposition",
        "Projecting the computed fields onto the output waveguide eigenmode basis provides modal amplitudes and powers."
    ),
    (
        "I need the Q and resonance frequency of an isolated optical cavity.",
        "eigenfrequency solver or time-domain ringdown",
        "A complex-frequency eigensolver can obtain resonance and Q directly; a time-domain decay calculation is an alternative."
    ),
    (
        "I need fields around a strongly scattering arbitrary 3D nanostructure over a broad wavelength range.",
        "3D FDTD",
        "The geometry is arbitrary, wavelength scale, and broadband information is required."
    ),
    (
        "I need only the first-order estimate of an MZI transfer spectrum from known arm phases and coupler matrices.",
        "analytical / scattering-matrix model",
        "A full Maxwell solution is unnecessary when validated component-level transfer matrices are already available."
    ),
    (
        "I need magneto-optic propagation in an anisotropic gyrotropic material.",
        "full-vector FEM or FDTD with tensor material support",
        "The solver must represent off-diagonal tensor constitutive parameters and vector polarization coupling."
    ),
]

for i in range(COUNTS["solver_selection"]):
    task, solver, why = solver_cases[i % len(solver_cases)]

    answer = (
        f"Recommended method: {solver}.\n"
        f"Reason: {why}"
    )

    add_record(
        task,
        answer,
        "solver_selection",
        "authored_solver_selection"
    )

print("Solver selection added:", COUNTS["solver_selection"])

Solver selection added: 120


In [ ]:
debug_cases = [
    (
        "My transmission changes by 25 percent when I double the mesh resolution.",
        "The result is not mesh converged.",
        "Perform a systematic convergence study. Refine the discretization until the quantity of interest changes below a predefined tolerance."
    ),
    (
        "I see a strong reflection coming back from the edge of my FDTD domain.",
        "The absorbing boundary or PML is inadequate or improperly configured.",
        "Increase separation from the structure, inspect PML thickness and settings, and verify that fields are not interacting with the outer boundary."
    ),
    (
        "My passive device reports transmitted plus reflected power equal to 1.27 times incident power.",
        "The normalization, monitor placement, modal projection, or convergence is probably incorrect.",
        "Check the normalization run, monitor orientation, modal normalization, source subtraction, absorption accounting, and mesh/time convergence."
    ),
    (
        "My ring spectrum looks flat even though the geometry should resonate.",
        "The spectral sampling or simulation duration may be insufficient, or the ring may not be coupled/excited correctly.",
        "Increase wavelength/frequency resolution, run long enough for a high-Q cavity, verify bus-ring coupling, and inspect whether the source excites the intended bus mode."
    ),
    (
        "My waveguide source launches a lot of radiation and several modes instead of TE0.",
        "The source profile is not sufficiently matched to the desired eigenmode.",
        "Use an eigenmode source or import the calculated TE0 field profile and ensure that the source plane spans the evanescent tails."
    ),
    (
        "My simulation predicts a resonance wavelength that shifts substantially when I change the domain padding.",
        "The boundaries are influencing the device.",
        "Increase padding, improve absorbing boundaries, and confirm that evanescent or radiative fields have decayed before reaching the boundaries."
    ),
    (
        "My 3D FDTD simulation requires hundreds of gigabytes for a centimeter-scale PIC.",
        "The numerical formulation is inappropriate for the scale of the complete problem.",
        "Use hierarchical simulation: full-wave models for local components, extract S-parameters, then simulate the large circuit with compact or scattering-matrix models."
    ),
    (
        "My mode solver returns many modes that look like radiation or box modes.",
        "The computational boundary or search region is also supporting non-guided numerical modes.",
        "Filter using effective index, confinement, symmetry and power-flow criteria, and enlarge or adjust the transverse domain when necessary."
    ),
    (
        "My result changes when I move a power monitor a tiny distance along a straight lossless waveguide.",
        "The monitor may be too close to a discontinuity or the result may include interference between forward and reflected fields.",
        "Move the monitor into a uniform region and use modal decomposition to separate forward and backward contributions."
    ),
    (
        "My silicon waveguide has almost the same effective index as air.",
        "The material assignment, units, wavelength, or geometry is likely wrong.",
        "Inspect material IDs, refractive-index units, geometric units, wavelength units, and whether the mode is actually confined to the intended core."
    ),
    (
        "A curved boundary gives noticeably different results when I translate the geometry by half a grid cell.",
        "Staircasing or grid discretization error is significant.",
        "Increase resolution or use subpixel/interface smoothing and verify translational convergence."
    ),
    (
        "My high-Q cavity decay has not reached a clean exponential before the simulation stops.",
        "The simulation time is too short or multiple modes are beating.",
        "Run longer, inspect the spectrum for nearby modes, and fit the decay only after transient excitation has sufficiently disappeared."
    ),
    (
        "I used symmetry boundaries and the desired mode completely vanished.",
        "The imposed symmetry is incompatible with the parity of the desired mode.",
        "Check the electric and magnetic parity of the mode and remove or correct the symmetry constraint."
    ),
    (
        "My wavelength sweep jumps irregularly between TE0 and TE1 mode solutions.",
        "The eigenmode search is changing mode ordering instead of tracking the same physical mode.",
        "Track modes using field overlap with the solution from the previous wavelength rather than relying only on eigenvalue ordering."
    ),
    (
        "My coupler simulation gives the right 50:50 ratio but substantial unexplained loss.",
        "Power may be radiating or entering higher-order modes, or the monitors may measure only one mode.",
        "Perform complete modal decomposition, measure radiated power, inspect transition regions, and verify conservation with all relevant channels included."
    ),
]

for i in range(COUNTS["debugging"]):
    symptom, diagnosis, fix = debug_cases[i % len(debug_cases)]

    q = (
        f"Debug this photonics simulation: {symptom}"
    )

    a = (
        f"Likely diagnosis: {diagnosis}\n"
        f"Recommended checks: {fix}"
    )

    add_record(
        q,
        a,
        "debugging",
        "authored_simulation_debugging"
    )

print("Debugging added:", COUNTS["debugging"])

Debugging added: 120


In [ ]:
def quantum_nonlinear_example():

    kind = random.choice([
        "purcell",
        "lifetime",
        "kerr",
        "fwm",
        "spdc",
        "cooperativity",
        "g2"
    ])

    if kind == "purcell":
        Q = random.randint(5_000, 2_000_000)
        Vnorm = rnd(0.3, 10, 3)

        Fp = 3/(4*math.pi**2) * Q/Vnorm

        q = (
            f"A resonant cavity has Q={Q} and normalized mode volume "
            f"V/((lambda/n)^3)={Vnorm}. Estimate the ideal Purcell factor."
        )

        a = (
            "For an ideally positioned, aligned, resonant emitter in the "
            "weak-coupling approximation,\n"
            "F_P = (3/(4*pi^2))*Q/V_norm.\n"
            f"F_P approximately {Fp:.3f}."
        )

    elif kind == "lifetime":
        lam_nm = random.choice([637, 780, 980, 1550])
        Q = random.randint(10_000, 2_000_000)

        omega = 2*math.pi*C/(lam_nm*1e-9)
        tau = Q/omega

        q = (
            f"A cavity at {lam_nm} nm has Q={Q}. "
            "Estimate the energy-decay lifetime using tau=Q/omega."
        )

        a = (
            "omega = 2*pi*c/lambda and tau=Q/omega.\n"
            f"tau approximately {tau*1e12:.3f} ps."
        )

    elif kind == "kerr":
        n2 = 10**random.uniform(-20, -17)
        intensity = 10**random.uniform(10, 14)

        dn = n2 * intensity

        q = (
            f"A Kerr medium has n2={n2:.3e} m^2/W and optical intensity "
            f"I={intensity:.3e} W/m^2. Estimate the nonlinear index change."
        )

        a = (
            "For the simple Kerr model, delta_n = n2*I.\n"
            f"delta_n = {dn:.3e}."
        )

    elif kind == "fwm":
        fp = rnd(180, 210, 3)
        fs = rnd(170, fp-1, 3)

        fi = 2*fp - fs

        q = (
            f"In degenerate four-wave mixing, the pump is {fp} THz "
            f"and the signal is {fs} THz. Estimate the idler frequency."
        )

        a = (
            "Energy conservation gives 2*f_p = f_s + f_i.\n"
            f"f_i = {fi:.3f} THz."
        )

    elif kind == "spdc":
        lp = random.choice([405, 532, 775])
        ls = rnd(lp*1.6, lp*2.4, 1)

        inv_li = 1/lp - 1/ls

        if inv_li <= 0:
            return quantum_nonlinear_example()

        li = 1/inv_li

        q = (
            f"In SPDC, a pump wavelength is {lp} nm and one daughter photon "
            f"has wavelength {ls} nm. Using energy conservation only, "
            "estimate the other daughter wavelength in vacuum."
        )

        a = (
            "Photon-energy conservation gives "
            "1/lambda_p = 1/lambda_s + 1/lambda_i.\n"
            f"lambda_i approximately {li:.2f} nm. "
            "Actual generation also requires momentum/phase matching."
        )

    elif kind == "cooperativity":
        g = rnd(0.1, 10, 3)
        kappa = rnd(1, 30, 3)
        gamma = rnd(0.1, 10, 3)

        Ccoop = 4*g*g/(kappa*gamma)

        q = (
            f"Using the convention C=4g^2/(kappa*gamma), calculate "
            f"cooperativity for g={g} GHz, kappa={kappa} GHz, "
            f"and gamma={gamma} GHz."
        )

        a = (
            "Using the stated convention,\n"
            "C = 4*g^2/(kappa*gamma).\n"
            f"C = {Ccoop:.4f}."
        )

    else:
        g2 = rnd(0.02, 1.5, 3)

        q = (
            f"A source has measured g^(2)(0)={g2}. "
            "What does this indicate about its photon statistics?"
        )

        if g2 < 0.5:
            interpretation = (
                "The value is below 0.5, indicating strong antibunching "
                "and behavior consistent with a predominantly single-photon emitter."
            )
        elif g2 < 1:
            interpretation = (
                "The value below 1 indicates antibunching/sub-Poissonian behavior, "
                "although it is not below the commonly used 0.5 single-emitter benchmark."
            )
        elif abs(g2-1) < 0.1:
            interpretation = (
                "The value is close to 1, approximately consistent with Poissonian "
                "coherent-light statistics."
            )
        else:
            interpretation = (
                "The value above 1 indicates photon bunching."
            )

        a = interpretation

    return q, a


for _ in range(COUNTS["quantum_nonlinear"]):
    q, a = quantum_nonlinear_example()

    add_record(
        q,
        a,
        "quantum_nonlinear",
        "synthetic_quantum_nonlinear"
    )

print("Quantum/nonlinear added:", COUNTS["quantum_nonlinear"])

Quantum/nonlinear added: 60


In [ ]:
def photonable_tool_example():

    device = random.choice([
        "ring_resonator",
        "waveguide",
        "mach_zehnder",
        "directional_coupler",
        "mmi",
        "bragg_grating"
    ])

    lam = random.choice([780, 1064, 1310, 1550])

    if device == "ring_resonator":
        radius = rnd(5, 50, 1)

        q = (
            f"Create a silicon ring resonator with radius {radius} um "
            f"and simulate transmission around {lam} nm."
        )

        obj = {
            "intent": "create_and_simulate",
            "device": "ring_resonator",
            "parameters": {
                "material": "Si",
                "radius_um": radius,
                "center_wavelength_nm": lam
            },
            "solver": "circuit_or_fullwave",
            "outputs": ["transmission_spectrum"],
            "validation": [
                "geometry_valid",
                "ports_defined",
                "solver_provenance_required"
            ]
        }

    elif device == "waveguide":
        width = rnd(0.35, 1.2, 3)
        height = rnd(0.18, 0.5, 3)

        q = (
            f"Create a silicon waveguide {width} um wide and {height} um high "
            f"and calculate its fundamental mode at {lam} nm."
        )

        obj = {
            "intent": "create_and_simulate",
            "device": "waveguide",
            "parameters": {
                "material": "Si",
                "width_um": width,
                "height_um": height,
                "wavelength_nm": lam
            },
            "solver": "eigenmode",
            "outputs": [
                "effective_index",
                "mode_profile"
            ],
            "validation": [
                "material_valid",
                "mesh_converged"
            ]
        }

    elif device == "mach_zehnder":
        delta = rnd(5, 100, 1)

        q = (
            f"Create an MZI with path difference {delta} um and calculate "
            f"its spectrum around {lam} nm."
        )

        obj = {
            "intent": "create_and_simulate",
            "device": "mach_zehnder",
            "parameters": {
                "path_difference_um": delta,
                "center_wavelength_nm": lam
            },
            "solver": "circuit",
            "outputs": ["transmission_spectrum"],
            "validation": [
                "splitters_defined",
                "ports_defined"
            ]
        }

    elif device == "directional_coupler":
        gap = rnd(0.1, 0.5, 3)

        q = (
            f"Create two coupled waveguides with gap {gap} um and determine "
            f"the 3 dB coupling length near {lam} nm."
        )

        obj = {
            "intent": "design",
            "device": "directional_coupler",
            "parameters": {
                "gap_um": gap,
                "wavelength_nm": lam
            },
            "solver": "supermode_eigenmode",
            "outputs": [
                "beta_even",
                "beta_odd",
                "estimated_3db_length"
            ],
            "validation": [
                "mode_identity_checked"
            ]
        }

    elif device == "mmi":
        q = (
            f"Create a 1x2 MMI splitter optimized around {lam} nm "
            "and return output imbalance and excess loss."
        )

        obj = {
            "intent": "design_and_optimize",
            "device": "mmi_1x2",
            "parameters": {
                "center_wavelength_nm": lam
            },
            "solver": "EME_or_FDTD",
            "outputs": [
                "output_1_power",
                "output_2_power",
                "imbalance_db",
                "excess_loss_db"
            ],
            "validation": [
                "power_conservation",
                "mesh_converged"
            ]
        }

    else:
        q = (
            f"Create a first-order Bragg grating around {lam} nm "
            "and calculate its reflection spectrum."
        )

        obj = {
            "intent": "create_and_simulate",
            "device": "bragg_grating",
            "parameters": {
                "target_wavelength_nm": lam
            },
            "solver": "FDTD_or_EME",
            "outputs": [
                "reflection_spectrum",
                "transmission_spectrum"
            ],
            "validation": [
                "period_defined",
                "power_conservation"
            ]
        }

    return q, json.dumps(obj, indent=2)


for _ in range(COUNTS["photonable_tools"]):
    q, a = photonable_tool_example()

    add_record(
        q,
        a,
        "photonable_tools",
        "photonable_experimental_schema"
    )

print("Photonable calls added:", COUNTS["photonable_tools"])

Photonable calls added: 60


In [ ]:
from collections import Counter

counts = Counter(x["category"] for x in records)

print("Total:", len(records))

for k, v in counts.items():
    print(f"{k:22s}: {v}")

assert len(records) == 1200

for category, expected in COUNTS.items():
    assert counts[category] == expected

print("\n✓ Dataset distribution correct")

Total: 1200
conceptual            : 300
numerical             : 240
pic_design            : 180
maxwell               : 120
solver_selection      : 120
debugging             : 120
quantum_nonlinear     : 60
photonable_tools      : 60

✓ Dataset distribution correct


In [ ]:
for example in random.sample(records, 10):

    print("\n" + "="*100)
    print("CATEGORY:", example["category"])
    print("\nQUESTION:")
    print(example["question"])
    print("\nANSWER:")
    print(example["answer"])


CATEGORY: debugging

QUESTION:
Debug this photonics simulation: My simulation predicts a resonance wavelength that shifts substantially when I change the domain padding.

ANSWER:
Likely diagnosis: The boundaries are influencing the device.
Recommended checks: Increase padding, improve absorbing boundaries, and confirm that evanescent or radiative fields have decayed before reaching the boundaries.

CATEGORY: conceptual

QUESTION:
Explain Bloch mode in integrated photonics.

ANSWER:
A Bloch mode is an eigenmode of a periodic electromagnetic structure and has the form of a periodic function multiplied by a phase factor exp(i k dot r).

CATEGORY: solver_selection

QUESTION:
I have a circuit containing hundreds of characterized MZIs, rings, and couplers and only need port-level spectra.

ANSWER:
Recommended method: circuit / scattering-matrix solver.
Reason: Using compact component S-parameters avoids an impractical full-wave simulation of the complete circuit.

CATEGORY: conceptual

QUES

In [ ]:
RAW_FILE = os.path.join(
    RUN_DIR,
    "photonics_training_raw.jsonl"
)

with open(RAW_FILE, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved:", RAW_FILE)

Saved: /content/drive/MyDrive/photonics_mistral_v01/photonics_training_raw.jsonl


In [ ]:
RAW_FILE = os.path.join(
    RUN_DIR,
    "photonics_training_raw.jsonl"
)

with open(RAW_FILE, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved:", RAW_FILE)

Saved: /content/drive/MyDrive/photonics_mistral_v01/photonics_training_raw.jsonl


In [ ]:
formatted_records = []

instruction_prefix = (
    "You are a specialist in photonics, optics, electromagnetic simulation, "
    "integrated photonics, and quantum optics. "
    "Give technically accurate answers. "
    "Do not invent simulation results. "
    "If a numerical result is requested, use the stated assumptions.\n\n"
)

for r in records:

    user_text = instruction_prefix + r["question"]

    prompt = tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": user_text
            }
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    completion = r["answer"] + tokenizer.eos_token

    formatted_records.append({
        "prompt": prompt,
        "completion": completion,
        "category": r["category"],
        "source_type": r["source_type"]
    })

print(formatted_records[0]["prompt"])
print("\n--- COMPLETION ---\n")
print(formatted_records[0]["completion"])

<s>[INST] You are a specialist in photonics, optics, electromagnetic simulation, integrated photonics, and quantum optics. Give technically accurate answers. Do not invent simulation results. If a numerical result is requested, use the stated assumptions.

Explain the physical meaning of effective index rather than only defining it.[/INST]

--- COMPLETION ---

The effective index n_eff relates a guided mode's propagation constant to the free-space wavenumber through beta = n_eff k0. It lies between values determined by the core and surrounding media and summarizes how the modal field experiences the structure.</s>


In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(formatted_records)

dataset = dataset.shuffle(seed=SEED)

split = dataset.train_test_split(
    test_size=0.08,
    seed=SEED
)

train_dataset = split["train"]
eval_dataset = split["test"]

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))

Training examples: 1104
Evaluation examples: 96


In [ ]:
train_dataset.save_to_disk(
    os.path.join(RUN_DIR, "train_dataset")
)

eval_dataset.save_to_disk(
    os.path.join(RUN_DIR, "eval_dataset")
)

print("Datasets saved to Drive.")

Saving the dataset (0/1 shards):   0%|          | 0/1104 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/96 [00:00<?, ? examples/s]

Datasets saved to Drive.


In [ ]:
benchmark_questions = [
    "A microring has radius 25 um and group index 4.0. Estimate its frequency FSR.",

    "Which solver should I use to calculate the effective index and TE0 mode profile of a uniform silicon waveguide?",

    "My passive FDTD device reports reflected plus transmitted power of 1.35. What should I investigate?",

    "Explain the difference between effective index and group index.",

    "How would you design a 50:50 directional coupler using supermode analysis?",

    "Why can a PML cause incorrect results if it is too close to a resonator?",

    "A cavity has Q=100000 and normalized mode volume V/(lambda/n)^3=1. Calculate the ideal Purcell factor.",

    "Create a Photonable instruction for a 10 um silicon ring at 1550 nm and request its transmission spectrum."
]


def ask_model(question, max_new_tokens=220):

    text = tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": instruction_prefix + question
        }],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    model.eval()

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )


baseline_results = {}

for q in benchmark_questions:

    answer = ask_model(q)

    baseline_results[q] = answer

    print("\nQUESTION:")
    print(q)

    print("\nBASE MODEL:")
    print(answer)

    print("\n" + "="*100)


QUESTION:
A microring has radius 25 um and group index 4.0. Estimate its frequency FSR.

BASE MODEL:
The Free Spectral Range (FSR) of a microring resonator can be calculated using the following formula:

FSR = c / (4 * π * radius * n_eff)

where:
- c is the speed of light in vacuum (approximately 3 x 10^8 m/s)
- π is the mathematical constant Pi (approximately 3.14159)
- radius is the radius of the microring (25 um)
- n_eff is the effective index of refraction of the mode in the microring (4.0 in this case)

Substituting the given values, we get:

FSR = c / (4 * π * radius * n_eff) = (3 * 10^8 m/s) / (4 * 3.14159 * 0.025 m * 4.0) = 15.70796


QUESTION:
Which solver should I use to calculate the effective index and TE0 mode profile of a uniform silicon waveguide?

BASE MODEL:
To calculate the effective index and TE0 mode profile of a uniform silicon waveguide, you can use a mode solver software that supports the finite-difference time-domain (FDTD) method or the finite element method (

In [20]:
with open(
    os.path.join(RUN_DIR, "baseline_results.json"),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        baseline_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Baseline saved.")

NameError: name 'RUN_DIR' is not defined

In [18]:
model.train()
model.config.use_cache = False

torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()

print(
    "Free VRAM:",
    round(free / 1024**3, 2),
    "GB"
)

Free VRAM: 9.46 GB


In [19]:
from trl import SFTConfig

training_args = SFTConfig(

    output_dir=os.path.join(
        RUN_DIR,
        "checkpoints"
    ),

    # First controlled experiment
    num_train_epochs=1.0,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=1e-4,

    lr_scheduler_type="cosine",

    warmup_steps=10,

    # Tesla T4
    fp16=True,
    bf16=False,

    gradient_checkpointing=True,

    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optim="paged_adamw_8bit",

    max_length=512,

    # Train only on expected answer
    completion_only_loss=True,

    logging_steps=5,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,

    save_total_limit=2,

    report_to="none",

    use_cache=False,

    shuffle_dataset=True,
)

print("Training config ready.")

NameError: name 'RUN_DIR' is not defined

In [46]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    processing_class=tokenizer
)

print("Trainer created.")

Trainer created.


In [ ]:
train_result = trainer.train()

NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [44]:
import os

FINAL_ADAPTER = os.path.join(
    RUN_DIR,
    "photonics_mistral_7b_lora_v01"
)

trainer.save_model(FINAL_ADAPTER)

tokenizer.save_pretrained(FINAL_ADAPTER)

print("✓ Adapter saved:")
print(FINAL_ADAPTER)

✓ Adapter saved:
/content/drive/MyDrive/photonics_mistral_v01/photonics_mistral_7b_lora_v01


In [47]:
metrics = trainer.evaluate()

print(metrics)

AttributeError: 'CausalLMOutputWithPast' object has no attribute 'last_hidden_state'

In [21]:
from google.colab import drive
drive.mount("/content/drive")

RUN_DIR = "/content/drive/MyDrive/photonics_mistral_v01"

Mounted at /content/drive


In [23]:
from datasets import load_from_disk

train_dataset = load_from_disk(
    f"{RUN_DIR}/train_dataset"
)

eval_dataset = load_from_disk(
    f"{RUN_DIR}/eval_dataset"
)

print(len(train_dataset), len(eval_dataset))

1104 96


In [27]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 83,886,080 || all params: 7,331,909,632 || trainable%: 1.1441


In [28]:
from collections import Counter

trainable_dtypes = Counter()

for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_dtypes[str(param.dtype)] += param.numel()

print(trainable_dtypes)

Counter({'torch.float32': 83886080})


In [29]:
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

print("Trainable parameters cast to FP32.")

Trainable parameters cast to FP32.


In [30]:
trainable_dtypes = Counter()

for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_dtypes[str(param.dtype)] += param.numel()

print(trainable_dtypes)

Counter({'torch.float32': 83886080})


In [31]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=f"{RUN_DIR}/checkpoints",

    num_train_epochs=1.0,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,

    # TESLA T4
    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optim="paged_adamw_8bit",

    max_length=512,

    completion_only_loss=True,

    logging_steps=5,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    report_to="none",
    use_cache=False,
)

In [32]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    processing_class=tokenizer,
)

print("Trainer ready.")

Trainer ready.


In [33]:
print("Model training mode:", model.training)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(
            "Example trainable parameter:",
            name,
            param.dtype,
            param.device
        )
        break

Model training mode: True
Example trainable parameter: base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16 cuda:0


In [34]:
import torch
from collections import Counter

# Cast ONLY trainable LoRA weights to FP32
for name, p in trainer.model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()

# Verify
dtypes = Counter()

for name, p in trainer.model.named_parameters():
    if p.requires_grad:
        dtypes[str(p.dtype)] += p.numel()

print("Trainable dtypes:", dtypes)

Trainable dtypes: Counter({'torch.float32': 83886080})


In [35]:
shown = 0

for name, p in trainer.model.named_parameters():
    if p.requires_grad:
        print(name, p.dtype, p.device, p.shape)
        shown += 1

        if shown >= 10:
            break

base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.float32 cuda:0 torch.Size([32, 4096])
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.float32 cuda:0 torch.Size([4096, 32])
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.float32 cuda:0 torch.Size([32, 4096])
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.float32 cuda:0 torch.Size([1024, 32])
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.float32 cuda:0 torch.Size([32, 4096])
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.float32 cuda:0 torch.Size([1024, 32])
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight t

In [43]:
trainer.args.max_steps = 1

test_result = trainer.train()

AttributeError: 'CausalLMOutputWithPast' object has no attribute 'last_hidden_state'

In [ ]:
TEST_DIR = f"{RUN_DIR}/one_step_test"

trainer.save_model(TEST_DIR)

print("Saved test adapter:", TEST_DIR)

Saved test adapter: /content/drive/MyDrive/photonics_mistral_v01/one_step_test


In [38]:
del trainer

torch.cuda.empty_cache()

In [39]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=f"{RUN_DIR}/checkpoints",

    num_train_epochs=1.0,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,

    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optim="adamw_torch",

    max_length=512,
    completion_only_loss=True,

    logging_steps=5,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    report_to="none",
    use_cache=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [40]:
for name, p in trainer.model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()

In [41]:
dtypes = Counter()

for name, p in trainer.model.named_parameters():
    if p.requires_grad:
        dtypes[str(p.dtype)] += p.numel()

print(dtypes)

Counter({'torch.float32': 83886080})


In [42]:
train_result = trainer.train()

AttributeError: 'CausalLMOutputWithPast' object has no attribute 'last_hidden_state'